<a href="https://colab.research.google.com/github/shivanshi-09/Introduction-to-Machine-Learning-/blob/main/A2Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.optimize import minimize
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
class SoftSVM:
  def __init__(self, C = 1.0, tol = 1e-5):
    self.C = C
    self.tol = tol
    self.alphas = None
    self.support_vectors = None
    self.sv_labels = None
    self.sv_alphas = None
    self.w = None
    self.b = None
  def build(self, X: np.ndarray, y:np.ndarray):
    n = X.shape[0]
    K = (y[:, None]*X)@(y[:, None]*X).T
    def neg_dual(alpha):
      return 0.5 * alpha @ K @ alpha - alpha.sum()
    def neg_dual_grad(alpha):
      return K @alpha - np.ones(n)
    constraints = {"type": "eq", "fun": lambda a: a @y, "jac": lambda a: y.astype(float),}
    bounds = [(0.0, self.C)]*n
    return neg_dual, neg_dual_grad, constraints, bounds, n
  def fit (self, X: np.ndarray, y: np.ndarray):
    X, y = np.array(X, dtype = float), np.array(y, dtype =float)
    obj, grad, constraints, bounds, n = self.build(X, y)
    alpha0 = np.zeros(n)
    result = minimize(
        fun = obj, x0 = alpha0, jac = grad, method = "SLSQP",
        constraints = constraints, bounds = bounds,
        options = {"maxiter": 10000, "ftol": 1e-9},
    )
    if not result.success:
      print(f" Optimizer didn't fully converege: {result.message}")
    self.alphas = result.x
    sv_mask = self.alphas >self.tol
    self.sv_alphas = self.alphas[sv_mask]
    self.support_vectors = X[sv_mask]
    self.sv_labels = y[sv_mask]
    n_sv = sv_mask.sum()
    print(f" Found {n_sv} support vectors out of {n} training samples")
    self.w = self.sv_alphas @ self.support_vectors

    margin_mask = (self.sv_alphas > self.tol) & (self.sv_alphas < self.C - self.tol)
    if margin_mask.sum() > 0:
        b_vals = (self.sv_labels[margin_mask]
                  - self.support_vectors[margin_mask] @ self.w)
        self.b = b_vals.mean()
    else:
        b_vals = self.sv_labels - self.support_vectors @ self.w
        self.b = b_vals.mean()

    return self
  def decision_function(self, X):
        return np.array(X, dtype=float) @ self.w + self.b

  def predict(self, X):
      return np.sign(self.decision_function(X))


In [ ]:
def decision_function(self, X:np.ndarray):
  return X @ self.w + self.b
def predict(self, X: np.ndarray):
  return np.sign(self.decision_function(np.array(X, dtype = float)))


In [ ]:
def breast_cancer_experiment():
  print("---------------------")
  print("Soft Margin SVM for Breast Cancer Dataset")
  print("---------------------")
  data = load_breast_cancer()
  X, y_raw = data.data, data.target
  y = np.where(y_raw == 1, 1, -1)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42, stratify=y)
  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  results = {}
  for C in [0.01, 0.1, 1.0, 10.0]:
    svm = SoftSVM(C=C, tol = 1e-5)
    svm.fit(X_train, y_train)
    train_preds = svm.predict(X_train)
    test_preds = svm.predict(X_test)
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)

    print(f" Train accuracy: {train_acc *100:2f}%")
    print(f" Test accuracy: {test_acc *100:2f}%")
    print(f" ||w|| : {np.linalg.norm(svm.w):.4f}")
    print(f" b : {svm.b:.4f}")
    results[C] = {
        "train_acc": round(train_acc * 100, 2),
            "test_acc" : round(test_acc  * 100, 2),
            "n_sv": int(len(svm.support_vectors)),
            "w_norm": round(float(np.linalg.norm(svm.w)), 4),
            "b": round(float(svm.b), 4),
            "report": classification_report(y_test, test_preds,
                               target_names=["Malignant", "Benign"]),
            "cm": confusion_matrix(y_test, test_preds).tolist(),
    }
  best_C = max(results, key = lambda c: results[c]["test_acc"])
  print(f"\n----------------------")
  print(f"  Best C = {best_C}  →  Test acc {results[best_C]['test_acc']}%")
  print(f"{'='*60}")
  print("\nClassification report for best C:")
  print(results[best_C]["report"])

  return results


if __name__ == "__main__":
    breast_cancer_experiment()


---------------------
Soft Margin SVM for Breast Cancer Dataset
---------------------
 Found 99 support vectors out of 455 training samples
 Train accuracy: 29.670330%
 Test accuracy: 35.087719%
 ||w|| : 0.5974
 b : -0.1135
 Found 51 support vectors out of 455 training samples
 Train accuracy: 57.802198%
 Test accuracy: 62.280702%
 ||w|| : 3.1400
 b : -0.9596
 Found 32 support vectors out of 455 training samples
 Train accuracy: 65.934066%
 Test accuracy: 62.280702%
 ||w|| : 25.0696
 b : -8.5289
 Found 26 support vectors out of 455 training samples
 Train accuracy: 72.087912%
 Test accuracy: 74.561404%
 ||w|| : 172.7398
 b : -91.6102

----------------------
  Best C = 10.0  →  Test acc 74.56%

Classification report for best C:
              precision    recall  f1-score   support

   Malignant       0.60      0.95      0.73        42
      Benign       0.96      0.62      0.76        72

    accuracy                           0.75       114
   macro avg       0.78      0.79      0.75  